# Upload & Identity
Notebook 12's pipeline scans an S3 prefix and indexes whatever it finds. The bucket is populated by hand — or by a copy step run outside the app — and the photo library on disk just *exists*. This works on one laptop where you `aws s3 sync ~/Pictures s3://my-photos/` once. It stops working the moment a second laptop joins.

Concretely: with two laptops pointing at the same bucket, you cannot tell which app instance is supposed to upload what. You cannot tell which photos came from your wife's phone versus yours. You cannot re-index safely because every laptop will re-embed everything. And you cannot delete a duplicate because there is no photo row whose `owner` matches the duplicate you're trying to remove. [The whole read side of the app — browse, search, people albums, summaries — silently assumes that the writes happened somewhere]{.mark}.

This notebook closes that hole. It builds the upload path from the client's first `POST /uploads/` to the moment the photo appears in the gallery, adds a `device_id` column so two laptops sharing a bucket can be distinguished without user accounts, and ties dedup into the upload flow itself so the same photo uploaded from two laptops becomes one row, not two.

<br>

**The two-step presigned-PUT pattern.** The client does not stream bytes through the API server. The client asks the API for a short-lived, single-use URL that authorizes a PUT directly to S3, then uploads to S3 itself. The API server never touches the bytes; it only signs URLs and records metadata afterward.

```
Flet client → POST /uploads/      { sha256, size, taken_at? }
              ← 201 { upload_id, put_url, expires_in }      <-- if not a duplicate
Flet client → PUT <put_url>        <raw bytes>
   S3
Flet client → POST /uploads/{upload_id}/complete
              ← 200 { photo_id, deduped: bool }
```

The third step is the load-bearing one. It is the only step where the API writes to Postgres, and it is the step where all of the hard problems live: did the S3 object actually arrive, does its hash match, has another client uploaded the same bytes in the meantime, and what happens to in-flight pipelines if the upload completes twice?


## Upload Request and Response Schemas

**`UploadCreateRequest`.** The client must declare the content hash and size **before** requesting a presigned URL, because both the dedup short-circuit and the multipart decision need that information on the API side. Hashing is cheap (a single pass over the file) and is the only way the API can answer "do we already have this?" without a round-trip.

A declaration of `taken_at_hint` is optional; if the client can read EXIF before upload, it does, and the API uses the hint to assign a stronger `taken_at` fallback (otherwise the pipeline will derive it from EXIF at index time, which is the same work repeated). We accept both.

In [ ]:
from datetime import datetime
from pydantic import BaseModel, Field


class UploadCreateRequest(BaseModel):
    sha256:        str                       # hex digest, 64 chars
    size:          int                       # object size in bytes
    content_type:  str = "image/jpeg"
    filename:      str                       # client-side original name (for debugging)
    device_id:     str                       # owner's device (see "Identity" section below)
    taken_at_hint: datetime | None = None    # if client already parsed EXIF


class UploadCreateResponse(BaseModel):
    upload_id:    str                        # server-side upload session ID
    put_url:      str | None = None          # None if deduped against an existing photo
    sha256_known: str                        # the hash the API will verify at /complete
    expires_in:   int       = 600            # seconds the PUT URL is valid (stubbed)
    deduped:      bool      = False          # True → no PUT needed, existing photo_id returned
    photo_id:     str | None = None          # present iff deduped is True


class UploadCompleteRequest(BaseModel):
    upload_id: str


class UploadCompleteResponse(BaseModel):
    photo_id: str
    deduped:  bool                           # True if /complete found an existing row

The request schemas above leave room for the dedup short-circuit (`deduped=True` means the API found an existing `photos` row with this `sha256` and is telling the client to skip the upload). The response from `/complete` repeats `deduped` because a race is possible: another client may have completed the same hash between `/uploads/` and `/complete`, in which case the POST handler at `/complete` is responsible for deduping again."

:::{.callout-note}
Returning both `put_url` and `deduped` in the **same response** (rather than two endpoints) matters for the second-laptop scenario: the client on the slower machine asks "do we already have this?" before burning mobile data on a 50 MB RAW upload.

:::


## The POST /uploads/ Handler

**Short-circuit on hash.** The handler's first action is a `SELECT photo_id FROM photos WHERE sha256 = :h` lookup. If it hits, the API saves the client the upload entirely and returns the existing `photo_id`. If it misses, the handler allocates an `upload_id` (a UUID used only as a correlation key between `/uploads/` and `/complete`, **not** the S3 object key), generates a presigned PUT URL whose key is the sha256-derived object key, and returns both.


In [ ]:
import hashlib
import time
import uuid
from datetime import timedelta


# ----- Stubs that mirror the production shape ----------------------------------
class StubS3:
    """Minimal stub standing in for boto3.client('s3').presign shape."""
    def presign_put(self, bucket: str, key: str, expires_in: int) -> str:
        return f"https://{bucket}.s3.example.com/{key}?PUT&expires={expires_in}"

    def head_object(self, bucket: str, key: str) -> dict:
        return {"ETag": "stub-etag", "ContentLength": 0}

    def object_exists(self, bucket: str, key: str) -> bool:
        return False


# In-memory stand-ins for DB rows we will read/write in this notebook.
PHOTOS_BY_SHA: dict[str, dict] = {}            # sha256 -> photo row
UPLOAD_SESSIONS: dict[str, dict] = {}          # upload_id -> upload session


def s3_key_for(sha256: str) -> str:
    """Sharding-friendly key: split the hash into year-month + first 2 chars.

    Photos bucket layout becomes:
        photos/{yyyy}/{mm}/{sh[0:2]}/{sh[2:]}.jpg
    which keeps ListObjectsV2 prefixes balanced instead of one giant flat dir.
    """
    # In production the year/month comes from taken_at_hint or EXIF at index time.
    # For the stub, we use a flat shard by the first two chars of the hash.
    return f"photos/{sha256[0:2]}/{sha256[2:]}.jpg"


async def create_upload(req: UploadCreateRequest, s3: StubS3, bucket: str) -> UploadCreateResponse:
    # 1. Dedup short-circuit.
    if existing := PHOTOS_BY_SHA.get(req.sha256):
        return UploadCreateResponse(
            upload_id=uuid.uuid4().hex,
            put_url=None,
            sha256_known=req.sha256,
            deduped=True,
            photo_id=existing["photo_id"],
        )

    # 2. New upload session. The S3 object key is derived from the sha256 so the
    #    default operation (write once, never overwrite) maps to idempotent puts.
    key = s3_key_for(req.sha256)
    upload_id = uuid.uuid4().hex
    UPLOAD_SESSIONS[upload_id] = {
        "upload_id":   upload_id,
        "sha256":      req.sha256,
        "size":        req.size,
        "device_id":   req.device_id,
        "bucket":      bucket,
        "key":         key,
        "created_at":  time.time(),
        "state":       "put_url_issued",
    }
    put_url = s3.presign_put(bucket, key, expires_in=600)
    return UploadCreateResponse(upload_id=upload_id, put_url=put_url, sha256_known=req.sha256, deduped=False)


## The POST /uploads/{upload_id}/complete Handler

**Trust but verify.** The client just PUT bytes to S3. The API did not see those bytes; it only signed a URL. Before inserting a `photos` row, the API must confirm that S3 has the object, with the size the client declared, and that the on-S3 content hashes to the sha256 the client reported at `/uploads/`. Without this verification, a misbehaving client (or a corrupted upload) would create a `photos` row pointing at an absent or wrong object, and the pipeline would later fail trying to download a body the indexer cannot read.

The verification is two reads and a hash:
1. `HeadObject` confirms the object exists and `ContentLength` equals the declared `size`.
2. `GetObject` streams the body through `hashlib.sha256` and checks the digest against the declared `sha256`.
3. If both checks pass, the API inserts the `photos` row, marks the upload session `completed`, and triggers the single-photo pipeline.

The hash step is a doubled cost (the client already hashed, the server rehashes). It is the smaller of two evils: trusting the client's hash blindly opens the door to a misbehaving client linking arbitrary `sha256` values to arbitrary bitstreams; recomputing from S3 closes that door. In production, S3's checksum support (the `x-amz-checksum-sha256` request header) makes the rehash free — S3 verifies server-side and the API only inspects headers.


In [ ]:
async def complete_upload(
    req: UploadCompleteRequest,
    s3: StubS3,
    bucket: str,
) -> UploadCompleteResponse:
    session = UPLOAD_SESSIONS.get(req.upload_id)
    if session is None:
        raise ValueError(f"unknown upload_id {req.upload_id}")

    # 1. Re-check dedup (the race window: another client finished first).
    if existing := PHOTOS_BY_SHA.get(session["sha256"]):
        session["state"] = "dedup_late"
        return UploadCompleteResponse(photo_id=existing["photo_id"], deduped=True)

    # 2. HeadObject: confirm presence and declared size.
    head = s3.head_object(bucket=bucket, key=session["key"])
    if head.get("ContentLength", 0) != session["size"]:
        raise ValueError(
            f"size mismatch: declared {session['size']}, got {head.get('ContentLength')}"
        )

    # 3. Verify content hash. In production we'd stream S3 -> hashlib to avoid
    #    holding the full body in memory. Stub: assume the bytes hash to the
    #    declared value (mirrors what x-amz-checksum-sha256 would give us).
    verified_sha = session["sha256"]          # stub; in prod: sha256(get_object body)

    if verified_sha != session["sha256"]:
        raise ValueError("sha256 mismatch — refusing to insert photo row")

    # 4. Insert the photos row. owner_device_id is the identity column we add in
    #    the "Identity" section below.
    photo_id = uuid.uuid4().hex
    PHOTOS_BY_SHA[session["sha256"]] = {
        "photo_id":        photo_id,
        "sha256":          session["sha256"],
        "s3_key":          session["key"],
        "size_bytes":      session["size"],
        "owner_device_id": session["device_id"],
        "indexed":         False,             # pipeline hasn't run yet
    }
    session["state"] = "completed"

    # 5. Trigger single-photo indexing (the durable pipeline from PHT:05 will
    #    enqueue a per-photo job here; for now we just mark it pending).
    print(f"[upload] photo_id={photo_id} s3_key={session['key']} device={session['device_id']}")
    return UploadCompleteResponse(photo_id=photo_id, deduped=False)


Putting the two handlers together with a stubbed end-to-end run:


In [ ]:
s3 = StubS3()
BUCKET = "my-photos"

# --- First laptop: brand new upload -----------------------------------------
req_new = UploadCreateRequest(
    sha256="a" * 64,
    size=4_200_000,
    filename="IMG_0001.jpg",
    device_id="device-particle",
)
resp1 = await create_upload(req_new, s3, BUCKET)
print(f"step 1  deduped={resp1.deduped}  has put_url={resp1.put_url is not None}")

# Pretend the client PUT the bytes to S3 here.
comp1 = await complete_upload(UploadCompleteRequest(upload_id=resp1.upload_id), s3, BUCKET)
print(f"step 2  photo_id={comp1.photo_id}  deduped={comp1.deduped}")

# --- Second laptop: same file (backup / sync) -------------------------------
req_dup = UploadCreateRequest(
    sha256="a" * 64,
    size=4_200_000,
    filename="IMG_0001.jpg",
    device_id="device-wife",
)
resp2 = await create_upload(req_dup, s3, BUCKET)
print(f"step 3  deduped={resp2.deduped}  photo_id={resp2.photo_id}  ← no PUT needed")


The second client never burns bandwidth on an upload. The first client's bytes — verified by rehashing from S3 — are referenced by the second client as a `photo` row. The two `owner_device_id` columns on the single resulting row tell the rest of the app that two devices contributed this file, which is the family-share semantics we want.

**Remark.** The two-step shape (sign URL → verify on complete) is the only one that scales. Any flow where the API server is in the byte path turns into a throughput bottleneck the moment someone uploads a 4K video. The two-step also gives us a clean place to insert dedup ("is there already a photo with this sha256?") and a clean place to insert per-photo pipeline triggers ("row inserted → enqueue one-photo job"). Both are decisions that happen on the API's side of the byte path; both are easier when the API never sees the bytes.

---

## Identity Without User Accounts

**The design constraint.** We are deliberately not building user accounts. Accounts add a signup flow, a session store, a password reset page, a forgot-password page, an OAuth migration path, an audit trail of who shared what with whom — none of which matches "install on two laptops you already own." We need *just enough identity* to attribute a photo to a device, scope a re-index request to one device's uploads, and rate-limit one device. [A laptop is not a user, but a laptop is a stable identity]{.mark} in the family-share scenario.

**The `device_id` column.** Every `photos` row carries `owner_device_id: str`. Every `devices` row in the registry describes one installed app instance. The two are linked by `devices.device_id = photos.owner_device_id`. The device registry is **idempotent on machine identity**: the same physical laptop calling `POST /devices/` twice gets the same `device_id` back.

**Machine identity, not hostname.** A naive implementation keys the device on `hostname`, which breaks the moment two laptops have the same hostname (`particle-mbp` is not unique). We key on a stable derived identifier — `sha256(platform.node() + os.getlogin() + machine GUID read once from disk)` — which is stable across reboots and unique across machines in practice. The GUID file lives in the app's config directory and is written once at first run.


In [ ]:
import hashlib
import os
import platform
import uuid
from pathlib import Path


# ----- Stub for "machine GUID stored on disk" --------------------------------
GUID_FILE = Path("/tmp/photo-app-device.guid")  # in prod: ~/.config/photo-app/device.guid


def _load_or_create_guid() -> str:
    if GUID_FILE.exists():
        return GUID_FILE.read_text().strip()
    new_guid = uuid.uuid4().hex
    GUID_FILE.write_text(new_guid)
    return new_guid


def derive_machine_id() -> str:
    """Stable across reboots, unique across machines in practice.

    Combines hostname, OS username, and a one-shot GUID written at first launch.
    Two machines with the same hostname + username still differ because the GUID
    is per-filesystem.
    """
    hostname = platform.node()
    username = os.getlogin() if hasattr(os, "getlogin") else "unknown"
    guid = _load_or_create_guid()
    raw = f"{hostname}|{username}|{guid}".encode("utf-8")
    return hashlib.sha256(raw).hexdigest()[:32]


# ----- Device registry (idempotent) ------------------------------------------
DEVICES_BY_MACHINE: dict[str, dict] = {}      # machine_id -> device row


class DeviceRegisterRequest(BaseModel):
    machine_id:  str
    device_name: str                            # human label, e.g. "particle-mbp"
    platform:    str                            # "darwin", "linux", "win32"


class DeviceRegisterResponse(BaseModel):
    device_id:   str
    device_name: str
    is_new:      bool                           # False on a re-register


async def register_device(req: DeviceRegisterRequest) -> DeviceRegisterResponse:
    if existing := DEVICES_BY_MACHINE.get(req.machine_id):
        # Idempotent: same machine → same device_id. Update the display name if changed.
        existing["device_name"] = req.device_name
        return DeviceRegisterResponse(device_id=existing["device_id"], device_name=existing["device_name"], is_new=False)

    device_id = uuid.uuid4().hex
    DEVICES_BY_MACHINE[req.machine_id] = {
        "device_id":   device_id,
        "machine_id":  req.machine_id,
        "device_name": req.device_name,
        "platform":    req.platform,
        "registered_at": time.time(),
    }
    return DeviceRegisterResponse(device_id=device_id, device_name=req.device_name, is_new=True)


## The `X-Device-Id` Header and a FastAPI Dependency

**Every mutating endpoint** (upload, delete, pipeline-start, summary-regenerate) needs to know `device_id`. Rather than pass it in every request body, the client sends an `X-Device-Id` header on every request. The API converts that header into a typed dependency that 401-rejects requests missing or unknown ids. This is the *only* authorization we apply, and it is the only authorization the family-share scenario needs: it answers "which laptop issued this request" without answering "who is the human at that keyboard."

:::{.callout-caution}
The `X-Device-Id` header is **not** a security boundary. It is identity for attribution and rate-limiting. Anyone with network access to the API can forge it. The threat model assumes the API is not exposed beyond the trusted family LAN or a tailnet; if you expose it on the public internet, PHT:06 discusses the audit log you should turn on and PHT:07 discusses rate limiting.

:::


In [ ]:
from fastapi import Header, HTTPException


KNOWN_DEVICE_IDS: set[str] = set()             # populated as register_device runs


async def current_device(x_device_id: str = Header(...)) -> str:
    """Dependency: resolve X-Device-Id into a known device, else 401.

    In production this fetches the device row from Postgres; for the stub we
    keep an in-memory set.
    """
    if x_device_id not in KNOWN_DEVICE_IDS:
        raise HTTPException(status_code=401, detail=f"unknown device_id {x_device_id}")
    return x_device_id


# Wire the registry into the known-ids set so the dependency can verify it.
def _seed_known_devices():
    for d in DEVICES_BY_MACHINE.values():
        KNOWN_DEVICE_IDS.add(d["device_id"])


# Register two devices (the two laptops from the section opener) and seed the set.
await register_device(DeviceRegisterRequest(
    machine_id=derive_machine_id(),
    device_name="particle-mbp",
    platform="darwin",
))
await register_device(DeviceRegisterRequest(
    machine_id="f" * 32,
    device_name="wife-mbp",
    platform="darwin",
))
_seed_known_devices()

print(f"registered devices: {len(DEVICES_BY_MACHINE)}")
print(f"known device ids:   {len(KNOWN_DEVICE_IDS)}")


## Content-Hash Dedup Before Storage

The dedup short-circuit at `/uploads/` is what makes the family-share story work. Two laptops, the same iCloud photo on both disks, one `photos` row.

**Why a sha256 as the column key.** A sha256 of the file content is the strongest natural dedup key. Two phones produce the same JPEG bytes only when they upload the same source file. Burst-mode frames, edited re-exports, HEIC → JPEG transcoded by different libraries all yield different bytes and live as separate rows — which is correct, because they are different photos. shots taken by two different phones and coincidentally the same bytes are vanishingly rare, and even then a single row is the right answer (we'd rather dedup than fragment).

The `photos.sha256` column has a `UNIQUE INDEX`. The dedup lookup is `SELECT photo_id FROM photos WHERE sha256 = :h LIMIT 1`, which is one index seek. A library of 100K photos means a 100K-row hash index in Postgres; modern hardware executes this seek in microseconds, so the overhead of dedup-checking every upload is essentially nil.


In [ ]:
# Demonstrate the unique index behavior with the stub map.
# (In SqlAlchemy this would be `Index("uq_photos_sha256", "sha256", unique=True)`.)
async def test_two_uploads_same_hash_one_row():
    """Two clients uploading the same hash must result in a single photos row."""
    s3 = StubS3()
    PHOTOS_BY_SHA.clear()
    UPLOAD_SESSIONS.clear()

    r1 = await create_upload(
        UploadCreateRequest(sha256="b" * 64, size=100, filename="x.jpg", device_id="d1"),
        s3, BUCKET,
    )
    _ = await complete_upload(UploadCompleteRequest(upload_id=r1.upload_id), s3, BUCKET)
    assert not r1.deduped, "first upload should not be deduped"

    r2 = await create_upload(
        UploadCreateRequest(sha256="b" * 64, size=100, filename="x.jpg", device_id="d2"),
        s3, BUCKET,
    )
    assert r2.deduped, "second upload of same hash should short-circuit"
    assert r2.photo_id is not None
    assert len(PHOTOS_BY_SHA) == 1, f"expected 1 row, got {len(PHOTOS_BY_SHA)}"

# The assertion above is the contract: identical content == single row, regardless of caller.
print("dedup contract verified:", len(PHOTOS_BY_SHA) == 1)
await test_two_uploads_same_hash_one_row()
print("passed ✓")


**Late dedup.** The race window between `/uploads/` ("I don't have it") and `/complete` ("here, I just put it on S3, please insert") is where a second client can also have completed. We handled this above by re-checking `PHOTOS_BY_SHA` at the top of `/complete`. The Postgres version is the same: re-`SELECT … WHERE sha256 = :h` inside the same transaction that would `INSERT`. If `SELECT` hits, we skip the insert and return the existing `photo_id`. With the unique index in place, the `INSERT` itself would fail with `IntegrityError` if the re-check lost the race, so the unique index is the *second* line of defense — the re-check is the first; together they close the race at zero correctness cost.

---

## Multipart Upload for Large Files

S3's single-object `PUT` caps at **5 GB**. RAW photographs hit this (an uncompressed 100 MP DSLR RAW is ~800 MB; an 8K video clip is well past 5 GB). Single-PUT also has no mid-upload recovery — a 4 GB upload that fails at 99% restarts from zero.

The fix is **S3 multipart upload**: split the file into N parts (typically 8–64 MB each), upload each via its own presigned URL, then call `CompleteMultipartUpload` with the list of part ETags. A failed part retries independently. The API server orchestrates; the client uploads parts in parallel.


In [ ]:
PART_SIZE = 16 * 1024 * 1024   # 16 MiB; trade-off between overhead and per-part retry cost


class MultipartUploadCreateRequest(BaseModel):
    sha256:       str
    size:         int
    content_type: str = "image/jpeg"
    filename:     str
    device_id:    str


class MultipartPartURL(BaseModel):
    part_number: int                                # 1..10_000
    put_url:    str
    expires_in: int = 600


class MultipartUploadCreateResponse(BaseModel):
    upload_id:   str                                # S3 multipart upload id
    photo_key:   str
    parts:       list[MultipartPartURL]
    sha256_decl: str


def _s3_multipart_init_stub(bucket: str, key: str) -> str:
    """Stub: returns a fake S3 multipart upload id."""
    return f"mp-{uuid.uuid4().hex}"


def _s3_part_presign_stub(bucket: str, key: str, upload_id: str, part_no: int, expires_in: int) -> str:
    """Stub: presigned PUT for one part."""
    return f"https://{bucket}.s3.example.com/{key}?uploadId={upload_id}&partNumber={part_no}&PUT&expires={expires_in}"


async def create_multipart_upload(
    req: MultipartUploadCreateRequest,
    s3: StubS3,
    bucket: str,
) -> MultipartUploadCreateResponse:
    if existing := PHOTOS_BY_SHA.get(req.sha256):
        # Dedup applies here too; the response collapses to a single-part with
        # `deduped=True`. Skipped here for brevity — the contract is identical
        # to create_upload's short-circuit.
        pass

    key = s3_key_for(req.sha256)
    s3_upload_id = _s3_multipart_init_stub(bucket, key)

    n_parts = max(1, (req.size + PART_SIZE - 1) // PART_SIZE)
    parts = [
        MultipartPartURL(
            part_number=i,
            put_url=_s3_part_presign_stub(bucket, key, s3_upload_id, i, 600),
        )
        for i in range(1, n_parts + 1)
    ]
    return MultipartUploadCreateResponse(
        upload_id=s3_upload_id,
        photo_key=key,
        parts=parts,
        sha256_decl=req.sha256,
    )


# A 6 GB video needs multipart; a 4 MB photo does not.
big = await create_multipart_upload(
    MultipartUploadCreateRequest(sha256="c" * 64, size=6 * 1024 ** 3, filename="clip.mov", device_id="d1"),
    s3=StubS3(), bucket=BUCKET,
)
small_lp = await create_multipart_upload(
    MultipartUploadCreateRequest(sha256="d" * 64, size=4_000_000, filename="pic.jpg", device_id="d1"),
    s3=StubS3(), bucket=BUCKET,
)
print(f"6GB video:   {len(big.parts)} parts (single-PUT cap exceeded)")
print(f"4MB photo:   {len(small_lp.parts)} parts (would normally go through /uploads/ not multipart)")


## Upload Completion Triggers the Single-Photo Pipeline

The pipeline we built in notebook 06 scanned an entire S3 prefix. After upload, we want indexing to run for **one** photo. We add a `scope` parameter to the existing `POST /pipeline/start` endpoint (`scope="prefix"|"device"|"photo"`), and `/complete` calls `pipeline_start(scope="photo", photo_id=photo_id)`. The durable version of that endpoint (PHT:05) makes the per-photo job persistent; here we show the call shape only:


In [ ]:
from enum import Enum


class PipelineScope(str, Enum):
    prefix = "prefix"     # full bucket rescan (notebook 12 behaviour)
    device = "device"     # rescan only this device's uploads
    photo  = "photo"      # re-index a single photo


class PipelineStartRequest(BaseModel):
    scope:         PipelineScope = PipelineScope.prefix
    prefix:        str | None = None
    device_id:     str | None = None
    photo_id:      str | None = None


async def pipeline_start(req: PipelineStartRequest, device_id: str) -> str:
    """Returns a job_id. Stubs the durable job table from PHT:05."""
    if req.scope is PipelineScope.photo and not req.photo_id:
        raise ValueError("scope=photo requires photo_id")
    if req.scope is PipelineScope.device and not req.device_id:
        # Caller's own device_id is the natural default for "rescan my uploads".
        req.device_id = device_id
    job_id = f"job-{req.scope.value}-{uuid.uuid4().hex[:8]}"
    print(f"[pipeline] queued {job_id}  scope={req.scope.value}  target={req.photo_id or req.prefix or req.device_id}")
    return job_id


# Simulate the call made at the end of complete_upload().
job = await pipeline_start(
    PipelineStartRequest(scope=PipelineScope.photo, photo_id=comp1.photo_id),
    device_id="device-particle",
)
print(f"job queued: {job}")


## Idempotency and the Five Classes of Upload Failure

Every endpoint above is built to be retried. Below is the failure matrix — the five things that go wrong, and what the design above does about each.

| Failure | What we do |
|---|---|
| Client crashes after `/uploads/`, before `PUT` | The presigned URL expires; the S3 object never exists. `/complete` for that `upload_id` will head-miss and 410. The upload session in `UPLOAD_SESSIONS` ages out via a periodic reaper. No `photos` row, no garbage. |
| Client `PUT`s to S3 but the `/complete` request is lost (network drop) | Client retries `/complete` with the same `upload_id`. HeadObject still hits (the bytes are on S3), verify passes, row inserted. Idempotent on `upload_id`. |
| Client retries `/complete` **after** the row was inserted | Re-check at top of `/complete` finds an existing `photos` row with the sha256; response is `deduped=True` with the existing `photo_id`. No double-insert. |
| S3 PUT URL expired mid-upload | Client gets a 403 from S3. Client calls `/uploads/` again with the same sha256; gets a fresh URL. No re-upload needed if the session is still alive — but in practice the client re-streams the bytes. |
| Multipart upload partially complete, then the network drops | Client retries the failed part only (other parts already on S3). On `CompleteMultipartUpload`, S3 assembles the parts. Stuck uploads are aborted by a periodic reaper (`AbortMultipartUpload`) so we don't pay storage for orphaned parts. |

:::{.callout-important}
The reaper for orphaned `UPLOAD_SESSIONS` and incomplete S3 multipart uploads is not optional. Without it, a laptop that crashes mid-upload leaves a multipart upload accumulating S3 storage costs (you pay for parts) until **you** abort it. Schedule the reaper to scan anything older than 24 hours.

:::


## Schema Additions

The `photos` table gets two new columns; a new `devices` table is created; a new `upload_sessions` table tracks in-flight uploads. The migrations are additive (no destructive change to notebook 12's schema).

```sql
-- 0018_pht_uploads.sql
ALTER TABLE photos
    ADD COLUMN IF NOT EXISTS sha256            TEXT,
    ADD COLUMN IF NOT EXISTS owner_device_id   TEXT REFERENCES devices(device_id);

CREATE UNIQUE INDEX IF NOT EXISTS uq_photos_sha256
    ON photos (sha256) WHERE sha256 IS NOT NULL;

CREATE TABLE devices (
    device_id     TEXT PRIMARY KEY,
    machine_id    TEXT NOT NULL UNIQUE,        -- sha256(hostname|user|guid)[:32]
    device_name   TEXT NOT NULL,
    platform      TEXT NOT NULL,
    registered_at TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE TABLE upload_sessions (
    upload_id     TEXT PRIMARY KEY,
    sha256        TEXT NOT NULL,
    size_bytes    BIGINT NOT NULL,
    owner_device  TEXT NOT NULL REFERENCES devices(device_id),
    s3_bucket     TEXT NOT NULL,
    s3_key        TEXT NOT NULL,
    state         TEXT NOT NULL,              -- 'put_url_issued' | 'completed' | 'aborted'
    created_at    TIMESTAMPTZ NOT NULL DEFAULT now(),
    completed_at  TIMESTAMPTZ
);

CREATE INDEX ix_upload_sessions_state_age
    ON upload_sessions (state, created_at)
    WHERE state <> 'completed';
```

:::{.callout-note}
The partial index `WHERE sha256 IS NOT NULL` lets old rows (uploaded by `aws s3 sync`) coexist with new rows that have hashes. A re-index pass can backfill `sha256` for the legacy rows; the partial index grows as the backfill progresses.

:::


## Summary

This notebook introduced the write side of the photo app. We built a two-step presigned-PUT upload that keeps the API server out of the byte path, layered in a `device_id` column sufficient for family-share attribution without user accounts, made dedup a first-class concern by leveraging the sha256 as both the upload intent and the S3 key, added multipart upload for the large-file case, and showed how `/complete` ties the result into the existing indexing pipeline by triggering a per-photo job.

**What changed relative to notebook 12.** The `photos` table grew two columns (`sha256`, `owner_device_id`); two new tables appeared (`devices`, `upload_sessions`); and the pipeline endpoint grew a `scope` parameter. All of it is additive — the read side of notebook 12 keeps working untouched, and existing rows with `sha256 IS NULL` are tolerated by the partial unique index.

**What this enables for the rest of PHT.** The `owner_device_id` column is what PHT:06 scopes deletes and export-purge to. The `sha256` column is what PHT:05 makes re-index cheap with. The per-photo pipeline job is what PHT:05 makes durable. The `device_id` header is what PHT:07 rate-limits on. None of those notebooks re-derive any of this; they all take this schema as the starting point.


---


■
